In [1]:
import gzip
import pickle
import pandas as pd
from gsp_python.gsp import GSP
from tslearn.piecewise import SymbolicAggregateApproximation as SAX

DEBUG:__init__:Creating converter from 7 to 5
DEBUG:__init__:Creating converter from 5 to 7
DEBUG:__init__:Creating converter from 7 to 5
DEBUG:__init__:Creating converter from 5 to 7


In [ ]:
with gzip.open("ts_normalized.pkl.gz", "rb") as f:
    ts_normalized = pickle.load(f)

sax = SAX(n_segments=200, alphabet_size_avg=5)

# Trasformazione: l'output è un array numpy 3D: (n_ts, n_segments, n_features)
ts_sax_array = sax.fit_transform(ts_normalized)

# Riconversione in lista di DataFrame per preservare la struttura e la leggibilità
ts = []
nomi_colonne = ts_normalized[0].columns

for matrice_ts in ts_sax_array:
    df_sax = pd.DataFrame(matrice_ts, columns=nomi_colonne)
    ts.append(df_sax)

colonne_da_rimuovere = [
    'id',
    'non-wear_flag',
    'battery_voltage',
    'weekday',
    'quarter',
    'relative_date_PCIAT',
    'X',
    'Y',
    'Z',
    # 'enmo',
    'anglez',
    'light',
    'sii_binary'
]

In [6]:
gsp_dataset = []

for df in ts:
    # Eliminazione vettoriale delle colonne non necessarie
    df.drop(columns=colonne_da_rimuovere, inplace=True, errors='ignore')

    gsp_sequence = []

    for index, row in df.iterrows():
        itemset = []

        for colonna in df.columns:
            valore_sax = row[colonna]

            if pd.notna(valore_sax):
                # Concatenazione del nome colonna al valore SAX (es. 'X_2')
                item = f"{colonna}_{valore_sax}"
                itemset.append(item)

        # Inserisce l'itemset solo se contiene almeno un elemento
        if itemset:
            gsp_sequence.append(itemset)

    # Inserisce la sequenza del paziente solo se non è vuota
    if gsp_sequence:
        gsp_dataset.append(gsp_sequence)

# Verifica della struttura ottenuta
print(f"Numero totale di sequenze (pazienti): {len(gsp_dataset)}")
if len(gsp_dataset) > 0 and len(gsp_dataset[0]) > 0:
    print(f"Struttura del primo itemset: {gsp_dataset[0][0]}")

Numero totale di sequenze (pazienti): 4437
Struttura del primo itemset: ['enmo_3']


In [ ]:
gsp = GSP(
    ds=gsp_dataset,
    minsup=0.8,
    mingap=0,
    maxgap=3,
    maxspan=16
)
patterns = gsp.run_gsp()

# Definizione della funzione di conversione estratta dal notebook


def output_to_dataframe(patterns):
    df = pd.DataFrame(data=patterns, columns=['pattern', 'support_count'])
    return df


# Generazione del DataFrame e ordinamento per frequenza decrescente
df_patterns = output_to_dataframe(patterns)
df_patterns = df_patterns.sort_values(
    by='support_count', ascending=False).reset_index(drop=True)

                                             pattern  support_count
0                                         [[enmo_2]]           4436
1                                         [[enmo_3]]           4427
2                                         [[enmo_4]]           4426
3  [[enmo_4], [enmo_4], [enmo_4], [enmo_4], [enmo...           4426
4                     [[enmo_4], [enmo_4], [enmo_4]]           4426
5           [[enmo_4], [enmo_4], [enmo_4], [enmo_4]]           4426
6                               [[enmo_4], [enmo_4]]           4426
7                                         [[enmo_1]]           4421
8                               [[enmo_2], [enmo_2]]           4419
9  [[enmo_4], [enmo_4], [enmo_4], [enmo_4], [enmo...           4417
